In [1]:
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Step 43 — The May 2026 car speed network: clipped to the study area, hourly link speeds by direction (`Car_speed_network_North.ipynb`)

A **car** speed network arrived on 5 October 2026 (`Input/CarSpeedData/GoogleSpeed_202605/`,
with the zip it was delivered in): one shapefile for the whole of Israel, with a measured
car speed per link, per hour of the day and per direction, for May 2026 — the same month as
the bus-speed street network of step 26 (§6x) and chosen for the same reason (a month clear of
the summer holiday distortion). It is the first car-speed source in the repository: the car
skim of steps 26 and 31 still rests on the 2017/18 survey's door-to-door times, and task C1 /
E5 (the 2026 car uplift) has been waiting on exactly this kind of data since 23 September
(`docs/NEXT_STEPS_HANDOVER_2026-09-23.md` §D item 8).

This step does the data preparation only, and answers three questions: **what does the layer
hold** (fields, units, coverage); **which links fall in the study area**, taking the TAZ
polygons of `Input/TAZ_North/TAZ_North.shp` (781 TAZs) as the reference, as every other
spatial step does; and **how are the hourly columns to be read**. The uplift itself (a car
skim between the V2 areas routed over these speeds) is the next step, not this one.

**Reading the columns** (user's instruction of 5 October 2026, verified below against the
`DIR` field): `SPD_{h}_AB` and `SPD_{h}_BA` hold the speed in km/h for the hour *h*:00 – *h*+1:00;
for 07:00–08:00 take `SPD_7_AB` and `SPD_7_BA`. `AB` is the direction of the link's drawn
geometry (from its first to its last vertex), `BA` the opposite direction — the same convention
as the bus network's `d_{weekday}_h_{hour}_{AB,BA}` (§1), except that there is no weekday
dimension here: one value per hour, for hours 6 to 21.

In [2]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
SRC = 'Input/CarSpeedData/GoogleSpeed_202605/GoogleSpeed.shp'
TAZ = 'Input/TAZ_North/TAZ_North.shp'
OUT = 'Output/car_speed'; FIG = 'Output/figures'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)
CRS = 'EPSG:2039'                                     # Israel TM Grid — both layers are delivered in it (two spellings of the same definition)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
assert not is_pointer(SRC), f'{SRC} is a Git LFS pointer'

t0 = time.time()
net = gpd.read_file(SRC, engine='pyogrio').set_crs(CRS, allow_override=True)
taz = gpd.read_file(TAZ, engine='pyogrio').set_crs(CRS, allow_override=True)
SPD = [c for c in net.columns if c.startswith('SPD_')]
HOURS = sorted({int(c.split('_')[1]) for c in SPD})
print(f'{len(net):,} links nationally, {net.geom_type.unique().tolist()}, {time.time() - t0:.1f} s; CRS {net.crs.to_epsg()}; {len(taz)} TAZ polygons')
print(f'fields: {[c for c in net.columns if c not in SPD and c != "geometry"]} + {len(SPD)} speed columns SPD_{{h}}_{{AB,BA}} for h = {HOURS[0]} … {HOURS[-1]}')
print(f'ID unique: {net["ID"].is_unique}; USERID unique: {net["USERID"].is_unique} ({net["USERID"].duplicated().sum()} duplicated rows); DIR values {sorted(net["DIR"].unique().tolist())}')
geom_km = net.geometry.length / 1000
print(f'LENGTH is in km: LENGTH / geometric length = {(net["LENGTH"] / geom_km).median():.4f} (median), total {net["LENGTH"].sum():,.0f} km')
has = net[SPD].notna().any(axis=1)
print(f'links with a speed record: {has.sum():,} of {len(net):,} ({has.mean():.1%}); the rest carry NaN in every speed column')

30,701 links nationally, ['LineString'], 0.8 s; CRS 2039; 781 TAZ polygons
fields: ['ID', 'DIR', 'LENGTH', 'USERID'] + 32 speed columns SPD_{h}_{AB,BA} for h = 6 … 21
ID unique: True; USERID unique: False (2 duplicated rows); DIR values [-1, 0, 1]
LENGTH is in km: LENGTH / geometric length = 1.0000 (median), total 14,882 km
links with a speed record: 14,630 of 30,701 (47.7%); the rest carry NaN in every speed column


## 1. The direction convention, checked against `DIR`

The bus network of step 26 reads `DIR` as 1 = one-way with the topology (only `AB` applies),
−1 = one-way against it (only `BA`), 0 = two-way. The table below cross-tabulates the 07:00
speeds of the links that have a record against `DIR`: a zero in the inapplicable direction is
the one-way marker, a zero in an applicable direction means **no observation in that hour**,
and the handful of links with zeros in both directions have a record but no observation at
all. The reading holds without exception, so downstream steps can take `DIR` from this layer
and treat `0` exactly as the bus network's "no observation" code.

In [3]:
rec = net[has].copy()
rows = []
for d in (-1, 0, 1):
    s = rec[rec['DIR'] == d]
    rows.append({'DIR': d, 'links_with_record': len(s),
                 'AB_zero': int((s['SPD_7_AB'] == 0).sum()), 'BA_zero': int((s['SPD_7_BA'] == 0).sum()),
                 'both_positive': int(((s['SPD_7_AB'] > 0) & (s['SPD_7_BA'] > 0)).sum()), 'both_zero': int(((s['SPD_7_AB'] == 0) & (s['SPD_7_BA'] == 0)).sum())})
dir_check = pd.DataFrame(rows).set_index('DIR'); print(dir_check.to_string())
assert dir_check.loc[1, 'both_positive'] == 0 and dir_check.loc[-1, 'both_positive'] == 0, 'a one-way link carries a speed in its closed direction: the DIR reading does not hold'
print(f"one-way links never carry a speed in the closed direction: DIR = 1 → AB only ({dir_check.loc[1, 'links_with_record'] - dir_check.loc[1, 'AB_zero']:,} observed), "
      f"DIR = −1 → BA only ({dir_check.loc[-1, 'links_with_record'] - dir_check.loc[-1, 'BA_zero']:,} observed); two-way: {dir_check.loc[0, 'both_positive']:,} with both directions observed")
dir_check.to_csv(f'{OUT}/car_speed_dir_convention_check.csv')
# how much do the speeds move across the day? (a per-link range over the 16 hours, links observed in every hour, AB direction)
ab = rec[[f'SPD_{h}_AB' for h in HOURS]]; ab = ab[(ab > 0).all(axis=1)]
rng = ab.max(axis=1) - ab.min(axis=1)
print(f'{len(ab):,} links observed in the AB direction in every hour: range of the hourly speed across the day — median {rng.median():.1f} km/h, p90 {rng.quantile(.9):.1f}, constant on {(rng < 1e-6).sum():,}; '
      f'mean AB speed 07:00 {ab["SPD_7_AB"].mean():.1f}, 12:00 {ab["SPD_12_AB"].mean():.1f}, 17:00 {ab["SPD_17_AB"].mean():.1f} km/h')

     links_with_record  AB_zero  BA_zero  both_positive  both_zero
DIR                                                               
-1                5049     5049       82              0         82
0                 5833      859      985           4072         83
1                 3748       62     3747              0         61
one-way links never carry a speed in the closed direction: DIR = 1 → AB only (3,686 observed), DIR = −1 → BA only (4,967 observed); two-way: 4,072 with both directions observed
8,660 links observed in the AB direction in every hour: range of the hourly speed across the day — median 2.3 km/h, p90 8.8, constant on 739; mean AB speed 07:00 37.2, 12:00 37.0, 17:00 37.2 km/h


## 2. The clip: links that touch the study area

The reference is the union of the 781 `TAZ_North` polygons. A link is kept when it
**intersects** that union, and it is kept **whole** — the geometry is not cut at the boundary,
because a network with links split at a polygon edge loses its topology (the new end points
join nothing) and the attributes of a split link (length, speed) would no longer describe
the piece. The cost of keeping whole links is measured below: the length of the retained links
that lies outside the polygons is a few tens of kilometres out of some five thousand. The
bounding-box selection step 26 used for the bus network is shown beside it for comparison;
the polygon rule drops the links in the bbox's empty corners (sea, and the strip south of the
study area's southern TAZs).

In [4]:
union = taz.geometry.union_all()
minx, miny, maxx, maxy = taz.total_bounds
in_bbox = net.cx[minx:maxx, miny:maxy]
north = net[net.intersects(union)].copy()
has_n = north[SPD].notna().any(axis=1)
outside_km = north.geometry.difference(union).length.sum() / 1000
inside_only = net.within(union).sum()
touched = taz[taz.intersects(north.geometry.union_all())]
print(f'bounding box of TAZ_North: {len(in_bbox):,} links; intersecting the TAZ polygons: {len(north):,} links, {north["LENGTH"].sum():,.0f} km '
      f'({inside_only:,} wholly inside; {outside_km:.0f} km of the kept links lies outside the polygons)')
print(f'with a speed record: {has_n.sum():,} links ({has_n.mean():.1%}), {north.loc[has_n, "LENGTH"].sum():,.0f} km ({north.loc[has_n, "LENGTH"].sum() / north["LENGTH"].sum():.1%} of the length); '
      f'TAZs touched by at least one link: {len(touched)} of {len(taz)}')
print('coverage by DIR in the study area:'); print(north.assign(record=has_n).groupby('DIR')['record'].agg(links='size', with_record='sum').to_string())
# the clipped layer, all 37 fields as delivered (an ESRI shapefile for the GIS, plus a GeoPackage for code)
north.to_file(f'{OUT}/GoogleSpeed_202605_North.shp', engine='pyogrio', encoding='UTF-8')
north.to_file(f'{OUT}/GoogleSpeed_202605_North.gpkg', layer='links', engine='pyogrio')
print(f'written {OUT}/GoogleSpeed_202605_North.shp (+ .shx/.dbf/.prj/.cpg) and .gpkg: {len(north):,} rows × {len(north.columns) - 1} fields')

bounding box of TAZ_North: 10,042 links; intersecting the TAZ polygons: 9,927 links, 4,996 km (9,874 wholly inside; 43 km of the kept links lies outside the polygons)
with a speed record: 4,693 links (47.3%), 3,994 km (79.9% of the length); TAZs touched by at least one link: 751 of 781
coverage by DIR in the study area:
     links  with_record
DIR                    
-1    2904         1295
 0    4025         2393
 1    2998         1005


written Output/car_speed/GoogleSpeed_202605_North.shp (+ .shx/.dbf/.prj/.cpg) and .gpkg: 9,927 rows × 36 fields


## 3. The hourly speeds as a long table

For analysis (and SQL), the 32 speed columns are unpivoted into one row per link, hour and
direction, keeping only the observed values: the inapplicable direction of a one-way link and
the zero "no observation" codes are dropped, so every row is a usable speed. `hour` is the
start of the hour (7 = 07:00–08:00), `direction` is `AB` or `BA` in the sense above, `DIR` and
`LENGTH_km` are carried from the link. A per-hour summary (links observed, kilometres, simple
and length-weighted mean speed, percentiles) sits beside it.

In [5]:
long = north.drop(columns='geometry').melt(id_vars=['ID', 'USERID', 'DIR', 'LENGTH'], value_vars=SPD, var_name='col', value_name='speed_kmh')
long['hour'] = long['col'].str.split('_').str[1].astype(int); long['direction'] = long['col'].str.split('_').str[2]
long = long[long['speed_kmh'] > 0].drop(columns='col').rename(columns={'LENGTH': 'LENGTH_km'})
long = long[['ID', 'USERID', 'DIR', 'LENGTH_km', 'hour', 'direction', 'speed_kmh']].sort_values(['ID', 'direction', 'hour']).reset_index(drop=True)
long.to_csv(f'{OUT}/car_link_speeds_hourly_north.csv', index=False, float_format='%.3f')
def summarise(g):
    w = g['LENGTH_km']
    return pd.Series({'links': len(g), 'km': w.sum(), 'mean_kmh': g['speed_kmh'].mean(), 'length_weighted_kmh': np.average(g['speed_kmh'], weights=w),
                      'p10_kmh': g['speed_kmh'].quantile(.1), 'p50_kmh': g['speed_kmh'].quantile(.5), 'p90_kmh': g['speed_kmh'].quantile(.9)})
summary = pd.DataFrame({k: summarise(g) for k, g in long.groupby(['hour', 'direction'])}).T
summary.index.names = ['hour', 'direction']; summary = summary.astype({'links': int}).reset_index()
summary.to_csv(f'{OUT}/car_speed_hourly_summary_north.csv', index=False, float_format='%.2f')
both = pd.DataFrame({k: summarise(g) for k, g in long.groupby('hour')}).T.astype({'links': int}); both.index.name = 'hour'
print(f'{len(long):,} observed link-hour-direction speeds on {long["ID"].nunique():,} links; per hour (both directions pooled):'); print(both.round(1).to_string())
am = long[long['hour'] == 7]
print(f'07:00–08:00: {len(am):,} directional speeds on {am["ID"].nunique():,} links, {am["LENGTH_km"].sum():,.0f} directional km; length-weighted mean {np.average(am["speed_kmh"], weights=am["LENGTH_km"]):.1f} km/h, median {am["speed_kmh"].median():.1f}')

101,840 observed link-hour-direction speeds on 4,640 links; per hour (both directions pooled):
      links      km  mean_kmh  length_weighted_kmh  p10_kmh  p50_kmh  p90_kmh
hour                                                                         
6      6365  5699.5      37.7                 51.7     17.6     31.5     68.1
7      6365  5699.5      37.8                 51.8     17.6     31.5     68.1
8      6365  5699.5      37.8                 51.8     17.6     31.5     68.2
9      6365  5699.5      37.7                 51.6     17.7     31.4     67.9
10     6365  5699.5      37.7                 51.6     17.6     31.4     68.1
11     6365  5699.5      37.7                 51.7     17.6     31.5     68.0
12     6365  5699.5      37.6                 51.6     17.5     31.4     67.7
13     6365  5699.5      37.6                 51.6     17.5     31.4     68.0
14     6365  5699.5      37.7                 51.6     17.6     31.5     68.0
15     6365  5699.5      37.7                 5

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for d, col in (('AB', BLUE), ('BA', ORANGE)):
    s = summary[summary['direction'] == d]
    ax[0].plot(s['hour'], s['length_weighted_kmh'], color=col, lw=2, marker='o', ms=4, label=f'{d}, length-weighted')
    ax[0].plot(s['hour'], s['p50_kmh'], color=col, lw=1, ls='--', label=f'{d}, median link')
ax[0].set_xlabel('hour of day (start of the hour)'); ax[0].set_ylabel('km/h'); ax[0].set_title('Car speed by hour, study-area links, May 2026'); ax[0].legend(frameon=False, fontsize=8)
ax[0].set_xticks(HOURS); ax[0].grid(color=GRID, lw=.6)
s = summary.groupby('hour')['links'].sum()
ax[1].bar(s.index, s.values, color=AQUA, width=.8); ax[1].set_xticks(HOURS); ax[1].set_xlabel('hour of day'); ax[1].set_ylabel('directional link speeds observed'); ax[1].set_title('Observations per hour'); ax[1].grid(axis='y', color=GRID, lw=.6)
for a in ax:
    for sp in ('top', 'right'): a.spines[sp].set_visible(False)
plt.tight_layout(); plt.savefig(f'{FIG}/car_speed_network_north_hourly_profile.png', dpi=150); plt.close()

# map: the clipped network, 07:00–08:00 speed (AB where observed, else BA), the unobserved links in grey
v = north['SPD_7_AB'].where(north['SPD_7_AB'] > 0, north['SPD_7_BA'])
fig, ax = plt.subplots(figsize=(9, 10))
taz.boundary.plot(ax=ax, color=AXIS, lw=.3)
north[~(v > 0)].plot(ax=ax, color=MUTED, lw=.4, alpha=.6)
norm = Normalize(vmin=10, vmax=90)
north[v > 0].plot(ax=ax, column=v[v > 0], cmap='RdYlGn', norm=norm, lw=1.0)
cb = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), ax=ax, shrink=.5, pad=.01); cb.set_label('km/h, 07:00–08:00 (AB, else BA)')
ax.set_title(f'Car speed network clipped to TAZ_North: {len(north):,} links, {int(has_n.sum()):,} with a May 2026 speed (grey: no record)', fontsize=10)
ax.set_axis_off(); plt.tight_layout(); plt.savefig(f'{FIG}/car_speed_network_north_map_0700.png', dpi=150); plt.close()
print('figures written')

figures written


## 4. What this layer is, and is not

Two checks place the layer against the networks already in the repository. It is **not the
Emme network with the traffic counts** (`Input/Network_with_Counts/`, step 36): the two have
a similar number of links nationally but no link end points in common, and the `ID` values
that coincide are a numerical accident. It **is derived from the bus-speed street network**
of step 26 (`Input/BusSpeedData/Streets`, 161,534 links): 95 % of the study-area links carry
a `USERID` that exists there, and in nearly every such case the street segment of that id
lies *on* the car link — the car layer is the same street geometry with consecutive segments
merged into longer links (a merged link is about 2.6 times its named segment), carrying the
`USERID` of one of its members. So the two layers share their geometry and their `DIR`
convention (once the merged link's drawing direction, which can be the reverse of its
segment's, is allowed for, `DIR` agrees on 99 % of the matched pairs), but `USERID` is
**not** a one-to-one key between them: a join on it finds one segment per car link, not the
car link's speed for every segment. A car skim between the V2
areas will be routed on this layer alone, with connectors from the area centroids to its
nearest nodes — as step 26 did on the bus network — and a link-level comparison of car and
bus speeds needs a spatial match, not the id.

In [7]:
try:
    import shapefile
    from shapely.geometry import LineString, Point
    rd = shapefile.Reader('Input/BusSpeedData/Streets/Streets')
    seg = {}
    for s, r in zip(rd.iterShapes(), rd.iterRecords()): seg.setdefault(r['USERID'], []).append((LineString(s.points), r['DIR']))
    found = on_link = same_ends = dir_agree = flipped = 0; ratio = []
    for row in north.itertuples():
        v = seg.get(row.USERID)
        if not v: continue
        found += 1; L = row.geometry
        for sl, sdir in v:
            a, b = Point(sl.coords[0]), Point(sl.coords[-1])
            if L.distance(a) < 2 and L.distance(b) < 2:
                on_link += 1; ratio.append(sl.length / L.length)
                same_ends += int(a.distance(Point(L.coords[0])) < 2 and b.distance(Point(L.coords[-1])) < 2)
                drawn_same_way = L.project(a) < L.project(b)          # the segment's vertex order vs the merged link's
                flipped += int(not drawn_same_way)
                dir_agree += int(sdir == (row.DIR if drawn_same_way else -row.DIR))
                break
    print(f'bus street network: {len(rd):,} segments; USERID found there for {found:,} of {len(north):,} study-area car links ({found / len(north):.0%}); '
          f'the named segment lies on the car link for {on_link:,} of them (identical end points {same_ends:,}; segment ÷ car-link length median {np.median(ratio):.2f}); '
          f'{flipped:,} merged links are drawn the opposite way to their segment; allowing for that, DIR agrees on {dir_agree:,} of {on_link:,} ({dir_agree / on_link:.1%})')
except Exception as ex:
    print('bus street network not readable here:', ex)
EMME = 'Input/Network_with_Counts/Emme_Links_Final_Res 2026-09-23.shp'
if not is_pointer(EMME):
    emme = gpd.read_file(EMME, engine='pyogrio', encoding='utf-8').to_crs(CRS)
    key = lambda g: {(round(l.coords[0][0]), round(l.coords[0][1]), round(l.coords[-1][0]), round(l.coords[-1][1])) for l in g}
    print(f'Emme network: {len(emme):,} links; link end-point pairs shared with this layer (1 m rounding): {len(key(net.geometry) & key(emme.geometry))}; ID values in common {len(set(net["ID"]) & set(emme["ID"])):,} (of {len(net):,}, accidental)')
else:
    print('Emme network is an LFS pointer here — the end-point check (0 shared pairs on 5 October 2026) is not repeated')

bus street network: 161,534 segments; USERID found there for 9,513 of 9,927 study-area car links (96%); the named segment lies on the car link for 9,435 of them (identical end points 2,672; segment ÷ car-link length median 0.38); 4,720 merged links are drawn the opposite way to their segment; allowing for that, DIR agrees on 9,353 of 9,435 (99.1%)


Emme network: 29,710 links; link end-point pairs shared with this layer (1 m rounding): 0; ID values in common 1,043 (of 30,701, accidental)


## 5. Outputs

- `Output/car_speed/GoogleSpeed_202605_North.{shp,shx,dbf,prj,cpg}` and `.gpkg` — the clipped
  layer: every link that intersects the `TAZ_North` polygons, whole, all 37 fields as delivered
  (`ID`, `DIR`, `LENGTH` in km, `USERID`, `SPD_6_AB` … `SPD_21_BA`).
- `Output/car_speed/car_link_speeds_hourly_north.csv` — the observed speeds, long:
  `ID, USERID, DIR, LENGTH_km, hour, direction, speed_kmh` (hour 6–21 = start of the hour;
  direction `AB` with / `BA` against the link geometry; zeros and closed directions dropped).
- `Output/car_speed/car_speed_hourly_summary_north.csv` — per hour and direction: links
  observed, km, mean, length-weighted mean, p10 / p50 / p90.
- `Output/car_speed/car_speed_dir_convention_check.csv` — the `DIR` × zero-speed table of §1.
- `Output/figures/car_speed_network_north_hourly_profile.png`, `car_speed_network_north_map_0700.png`.

**Limits.** Half the links in the study area carry no speed record at all (NaN in every
column) — four fifths of the length is covered, so the gaps are the short local links — and
the record is a single value per hour for the month: no weekday dimension, no spread, and
exactly the same set of directional links observed in every one of the 16 hours. The hourly
profile is flat: on the links observed in every hour the speed moves by a median of about
2 km/h across the day, far less than the peak / off-peak contrast a road user would expect,
so the values look like a monthly aggregate lightly modulated by hour rather than
hour-specific measurements; the provider's method is not documented with the delivery and
should be asked for before the 07:00 speeds are used as "peak" speeds. Speeds are not
available before 06:00 or after 22:00.